# 🔬 DPS Global Conditional — CLIP-MMD Branch
Diffusion Posterior Sampling with CLIP-MMD guidance using SDXL Turbo + ControlNet.
Differences from `main` branch:
- Target distribution represented in **CLIP embedding space** (768-dim) instead of raw latent space
- MMD computed with standard unbiased estimator
- Target prompts: **man / woman portraits**
- `generate_and_store_cs` supports configurable `controlnet_conditioning_scale`

## 1. Environment Setup & GitHub Clone

In [ ]:
! pip install controlnet_aux

In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb
from diffusers import DDIMScheduler

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "scribble_cond_loss"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/SD_cond_SD_controlnet"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

wandb_token = userdata.get('WANDB')
if wandb_token:
    wandb.login(key=wandb_token)
else:
    wandb.login()

## 2. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import gc
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from IPython.display import display
import wandb

from models       import load_models, setup_gradient_checkpointing
from image_utils  import sobel_proxy, latent_to_pil, build_base_image
from clip_utils   import load_clip_model, encode_images_clip
from generation   import (
    generate_and_store,
    compute_pred_x0_direct,
    generate_and_store_cs,
    predict_noise_cfg,
    compute_pred_x0,
    denoise_step,
    run_dps_step,
    run_dps_step_clip,
)
from visualization import plot_row, visualize_step
from metrics       import compute_mmd, evaluate_distribution_mmd

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

## 3. Load Models

In [ ]:
architect, sprinter = load_models(device)
architect.scheduler = DDIMScheduler.from_config(architect.scheduler.config)
print("✅ Diffusion models loaded.")

## 4. Load CLIP Model

In [ ]:
clip_model, clip_processor = load_clip_model(device)
print("✅ CLIP model loaded and frozen.")

## Build Base Image & Sobel Conditioning

In [ ]:
base_image_pil, base_tensor = build_base_image(device)

with torch.no_grad():
    sobel_cond_tensor = sobel_proxy(base_tensor, device)
    sobel_cond_pil    = T.ToPILImage()(sobel_cond_tensor.squeeze(0).cpu())

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(base_image_pil);                    axes[0].set_title("Base Shape");        axes[0].axis('off')
axes[1].imshow(sobel_cond_pil, cmap='gray');       axes[1].set_title("Sobel Conditioning"); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 6. Generate Target Distributions (Man / Woman Portraits)

In [ ]:
N       = 20
n_total = N // 2

CONTROLNET_SCALE = 0.5   # reduced from 1.0 — less rigid shape constraint

with torch.no_grad():
    man_images, man_latents = generate_and_store_cs(
        sprinter,
        "a superrealistic portrait photograph of a man, studio lighting",
        sobel_cond_pil, n_total, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )
    woman_images, woman_latents = generate_and_store_cs(
        sprinter,
        "a superrealistic portrait photograph of a woman, studio lighting",
        sobel_cond_pil, n_total, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )

all_latents = np.vstack([man_latents, woman_latents])
print(f"✅ Target latents shape: {all_latents.shape}")

plot_row(man_images,   "Man Portrait Samples")
plot_row(woman_images, "Woman Portrait Samples")


## Encode Target Distributions into CLIP Space

In [ ]:
def pil_images_to_tensor(pil_list, device):
    """Convert list of PIL images to [B, 3, H, W] float tensor in [0,1]"""
    tensors = [TF.to_tensor(img).unsqueeze(0) for img in pil_list]
    return torch.cat(tensors, dim=0).to(device)

with torch.no_grad():
    man_pixel_tensor   = pil_images_to_tensor(man_images,   device)
    woman_pixel_tensor = pil_images_to_tensor(woman_images, device)

    man_clip_embs   = encode_images_clip(man_pixel_tensor,   clip_model, clip_processor)  # [n_total, 768]
    woman_clip_embs = encode_images_clip(woman_pixel_tensor, clip_model, clip_processor)  # [n_total, 768]

all_clip_embeddings = torch.cat([man_clip_embs, woman_clip_embs], dim=0)  # [N, 768]
print(f"✅ Target CLIP embeddings shape: {all_clip_embeddings.shape}")

# ── Sanity checks ──────────────────────────────────────────────────────────────
assert all_clip_embeddings.shape == (N, 768), f"Bad shape: {all_clip_embeddings.shape}"

norms = all_clip_embeddings.norm(dim=-1)
assert torch.allclose(norms, torch.ones(N, device=device), atol=1e-3), "Not normalized!"
print(f"   Norms min/max: {norms.min():.4f} / {norms.max():.4f}")

intra_sim = (man_clip_embs @ man_clip_embs.T).mean().item()
inter_sim = (man_clip_embs @ woman_clip_embs.T).mean().item()
print(f"   Intra-class cosine sim (man↔man)   : {intra_sim:.4f}")
print(f"   Inter-class cosine sim (man↔woman) : {inter_sim:.4f}")
assert intra_sim > inter_sim, "Classes not separated in CLIP space — something is wrong!"
print("✅ Classes are separable in CLIP space")

# PCA in CLIP space
_pca    = PCA(n_components=2)
_coords = _pca.fit_transform(all_clip_embeddings.cpu().numpy())
plt.figure(figsize=(8, 6))
plt.scatter(_coords[:n_total, 0], _coords[:n_total, 1], c='dodgerblue', label='Man',   alpha=0.7)
plt.scatter(_coords[n_total:, 0], _coords[n_total:, 1], c='crimson',    label='Woman', alpha=0.7)
plt.title("PCA of Target CLIP Embeddings")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
del _pca, _coords

## Config & Scribble Seed Image

In [ ]:
from controlnet_aux import HEDdetector

hed = HEDdetector.from_pretrained("lllyasviel/Annotators")

# Use the first man portrait as the source scribble
source_image = man_images[2]
scribble_pil = hed(source_image, scribble=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(source_image); axes[0].set_title("Source Portrait");  axes[0].axis('off')
axes[1].imshow(scribble_pil); axes[1].set_title("HED Scribble");     axes[1].axis('off')
plt.tight_layout(); plt.show()

# Replace sobel_cond_pil with the HED scribble for all downstream use
sobel_cond_pil = scribble_pil
print("✅ HED scribble ready — sobel_cond_pil updated.")


In [ ]:

sprinter.vae.to(dtype=torch.float16)

with torch.no_grad():
    results = sprinter(
        prompt=["a superrealistic portrait photograph, studio lighting"] * 5,
        negative_prompt=[""] * 5,
        image=[scribble_pil] * 5,
        num_inference_steps=2,
        guidance_scale=0.0,
        controlnet_conditioning_scale=0.5,
    )

sprinter.vae.to(dtype=torch.float32)

plot_row(results.images, "Sprinter samples from HED scribble", count=5)
cond_images = results.images  # saved for CLIP PCA in section 8


In [ ]:
prompt               = ''
negative_prompt      = ''
guidance_scale       = 0.0
height, width        = 512, 512
n_steps              = 30
start_step           = 15
num_variations       = N
variation_batch_size = 1
base_zeta_prime      = 1.0
n_eval               = 10
eval_interval  = max(1, (n_steps - start_step) // 5)  # ~5 checkpoints

sprinter_variation_prompt    = 'a superrealistic professional photograph of'
sprinter_target_man_prompt   = 'a superrealistic portrait photograph of a man, studio lighting'
sprinter_target_woman_prompt = 'a superrealistic portrait photograph of a woman, studio lighting'
sprinter_eval_prompt         = 'a superrealistic professional photograph of'

print(f"Prompt     : '{prompt}'")
print(f"Steps      : {n_steps}")
print(f"Variations : {num_variations}/step")
print(f"Zeta base  : {base_zeta_prime}")
print(f'n_eval     : {n_eval}')

run = wandb.init(
    project='measure_MMD_between_uncond_dps',
    entity='conditional-matching',
    config={
        'prompt':           prompt,
        'negative_prompt':  negative_prompt,
        'n_targets':        N,
        'n_steps':          n_steps,
        'start_step':       start_step,
        'strength':         1 - start_step / n_steps,
        'steps_run':        n_steps - start_step,
        'scheduler_type':   type(architect.scheduler).__name__,
        'num_variations':   num_variations,
        'base_zeta':        base_zeta_prime,
        'guidance_scale':   guidance_scale,
        'controlnet_scale': CONTROLNET_SCALE,
        'edge_method':      'hed_scribble',
        'n_eval':           n_eval,
        'eval_interval':  eval_interval,

        'architect_model':  architect.config._name_or_path,
        'sprinter_model':   sprinter.config._name_or_path,
        'sprinter_variation_prompt':    sprinter_variation_prompt,
        'sprinter_target_man_prompt':   sprinter_target_man_prompt,
        'sprinter_target_woman_prompt': sprinter_target_woman_prompt,
        'sprinter_eval_prompt':         sprinter_eval_prompt,
    },
)
print(f'✅ wandb run: {run.name}')

In [ ]:
N       = N
n_total = N // 2

CONTROLNET_SCALE = 0.5   # reduced from 1.0 — less rigid shape constraint

with torch.no_grad():
    man_images, man_latents = generate_and_store_cs(
        sprinter,
        sprinter_target_man_prompt,
        sobel_cond_pil, n_total, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )
    woman_images, woman_latents = generate_and_store_cs(
        sprinter,
        sprinter_target_woman_prompt,
        sobel_cond_pil, n_total, batch_size=2, cn_scale=CONTROLNET_SCALE,
    )

all_latents = np.vstack([man_latents, woman_latents])
print(f"✅ Target latents shape: {all_latents.shape}")

plot_row(man_images,   "Man Portrait Samples")
plot_row(woman_images, "Woman Portrait Samples")


In [ ]:
# CHANGED: new cell — log all input images before the loop
wandb.log({
    'scribble':             wandb.Image(scribble_pil),
    'source_portrait':      wandb.Image(source_image),
    'target_samples_man':   [wandb.Image(p) for p in man_images],
    'target_samples_woman': [wandb.Image(p) for p in woman_images],
})
print('✅ Input images logged to wandb.')

## Prepare for DPS Loop

In [ ]:
sprinter.vae.to(dtype=torch.float32)
setup_gradient_checkpointing(architect, sprinter)

with torch.no_grad():
    (
        prompt_embeds,
        negative_prompt_embeds,
        pooled_prompt_embeds,
        negative_pooled_prompt_embeds,
    ) = architect.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt,
        device=device,
        do_classifier_free_guidance=True,
        num_images_per_prompt=1,
    )

architect.scheduler.set_timesteps(n_steps, device=device)
timesteps = architect.scheduler.timesteps

import copy
scheduler_regular = copy.deepcopy(architect.scheduler)

add_time_ids = torch.tensor(
    [[height, width, 0, 0, height, width]], dtype=prompt_embeds.dtype, device=device
)
added_cond_kwargs = {
    "text_embeds" : torch.cat([negative_pooled_prompt_embeds, pooled_prompt_embeds], dim=0),
    "time_ids"    : add_time_ids.repeat(2, 1),
}
cfg_encoder_states = torch.cat([negative_prompt_embeds, prompt_embeds], dim=0)


# ── SDEdit-style init: encode scribble → latent, noise to start_step ────────
with torch.no_grad():
    scribble_tensor = TF.to_tensor(scribble_pil).unsqueeze(0).to(device).to(torch.float32)
    scribble_tensor = (scribble_tensor * 2.0) - 1.0  # [0,1] → [-1,1]
    scribble_latent = architect.vae.encode(scribble_tensor).latent_dist.mean
    scribble_latent = scribble_latent * architect.vae.config.scaling_factor

start_step     = 15  # run DPS from this step onwards (halfway through)
t_start        = timesteps[start_step]
alphas_cumprod = architect.scheduler.alphas_cumprod.to(device)
alpha          = alphas_cumprod[t_start.long()].to(torch.float32)
noise          = torch.randn_like(scribble_latent)
latents        = ((alpha ** 0.5) * scribble_latent + ((1 - alpha) ** 0.5) * noise).to(torch.float16)
latents_regular = latents.detach().clone()

# Only run the timesteps from start_step onwards
timesteps_to_run = timesteps[start_step:]

step_gradients = []
step_vis_data  = []

print(f"✅ Ready. Starting from step {start_step}/{n_steps}  (t={t_start.item():.0f})")
print(f"   Running {len(timesteps_to_run)} DPS steps...")

## DPS Denoising Loop (CLIP-MMD)

In [ ]:
print(f'Beginning CLIP-MMD DPS ({n_steps} steps, {num_variations} variations/step)...')
pbar = tqdm(enumerate(timesteps_to_run), total=len(timesteps_to_run), desc='DPS')

eval_interval = max(1, len(timesteps_to_run) // 5)

for i, t in pbar:
    print(f"\n{'='*60}")
    print(f'Step {i+1}/{len(timesteps_to_run)}  (t={t})')
    print(f"{'='*60}")

    latents_step         = latents.detach().requires_grad_(True)
    latents_step_regular = latents_regular.detach()

    noise_pred = predict_noise_cfg(
        architect.unet, architect.scheduler,
        latents_step, t, cfg_encoder_states, added_cond_kwargs, guidance_scale
    )
    with torch.no_grad():
        noise_pred_regular = predict_noise_cfg(
            architect.unet, scheduler_regular,
            latents_step_regular, t, cfg_encoder_states, added_cond_kwargs, guidance_scale
        )

    pred_x0 = compute_pred_x0_direct(architect.scheduler, noise_pred, t, latents_step)
    with torch.no_grad():
        pred_x0_regular = compute_pred_x0_direct(scheduler_regular, noise_pred_regular, t, latents_step_regular)

    pred_x0_scaled = pred_x0 / architect.vae.config.scaling_factor
    def vae_decode_checkpoint(lat):
        return architect.vae.decode(lat.to(architect.vae.dtype)).sample
    pixel_x0      = torch.utils.checkpoint.checkpoint(vae_decode_checkpoint, pred_x0_scaled, use_reentrant=False)
    pixel_x0_norm = torch.clamp((pixel_x0 + 1.0) / 2.0, 0.0, 1.0)

    grad, mmd_loss, zeta_i, loss_norm, vl_clip_flat = run_dps_step_clip(
        latents=latents, latents_step=latents_step, noise_pred=noise_pred,
        pixel_x0_norm=pixel_x0_norm, sprinter=sprinter,
        all_clip_embeddings=all_clip_embeddings,
        num_variations=num_variations, variation_batch_size=variation_batch_size,
        base_zeta_prime=base_zeta_prime, clip_model=clip_model,
        clip_processor=clip_processor, vae=sprinter.vae,
        vae_scaling_factor=sprinter.vae.config.scaling_factor,
        variation_prompt=sprinter_variation_prompt,
    )

    grad_norm = grad.norm().item()
    zeta_val  = zeta_i.item() if isinstance(zeta_i, torch.Tensor) else zeta_i
    print(f'  MMD={mmd_loss.item():.6f}  ζi={zeta_val:.4f}  ∥∇∥={grad_norm:.6f}')

    if torch.isnan(grad).any():
        print(f'  ⚠️  NaN in gradient at step {i} — skipping correction')
        correction = torch.zeros_like(latents_step)
    else:
        correction = -zeta_i * grad

    step_gradients.append({
        'step':            i,
        'timestep':        t.item(),
        'gradient_norm':   grad_norm,
        'mmd_loss':        mmd_loss.item(),
        'zeta_i':          zeta_val,
        'loss_norm':       loss_norm.item(),
        'correction_norm': zeta_val * grad_norm,
    })

    wandb_log = {
        'step':            i,
        'mmd_loss':        mmd_loss.item(),
        'gradient_norm':   grad_norm,
        'zeta':            zeta_val,
        'correction_norm': zeta_val * grad_norm,
    }

    # CHANGED: intermediate unguided conditional MMD — placed AFTER run_dps_step_clip
    if i % eval_interval == 0:
        unguided_mmd, _, _ = evaluate_distribution_mmd(
            pred_x0_regular.detach(), architect.vae, architect.image_processor,
            sprinter, clip_model, clip_processor,
            all_clip_embeddings, sprinter_eval_prompt, n_eval=n_eval, device=device,
        )
        wandb_log['intermediate/unguided_cond_mmd'] = unguided_mmd
        wandb_log['intermediate/guided_cond_mmd']   = mmd_loss.item()
        wandb_log['intermediate/cond_mmd_delta']    = mmd_loss.item() - unguided_mmd
        print(f'  [eval] guided={mmd_loss.item():.6f}  unguided={unguided_mmd:.6f}  delta={mmd_loss.item()-unguided_mmd:.6f}')

    wandb.log(wandb_log)

    with torch.no_grad():
        sd = {
            'step':                     i,
            'timestep':                 t.item(),
            'mmd_loss':                 mmd_loss.item(),
            'zeta_i':                   zeta_val,
            'latents_step_cpu':         latents_step.detach().cpu(),
            'latents_step_regular_cpu': latents_step_regular.detach().cpu(),
            'pred_x0_cpu':              pred_x0.detach().cpu(),
            'pred_x0_regular_cpu':      pred_x0_regular.detach().cpu(),
            'variation_clip_flat':      vl_clip_flat,
        }
        step_vis_data.append(sd)

    visualize_step(sd, architect, sprinter, all_clip_embeddings.cpu().numpy())

    latents = denoise_step(architect.scheduler, noise_pred, t, latents_step, correction=correction)
    with torch.no_grad():
        latents_regular = denoise_step(scheduler_regular, noise_pred_regular, t, latents_step_regular)

    del grad, mmd_loss, loss_norm, zeta_i, correction
    del pixel_x0, pixel_x0_norm, pred_x0, pred_x0_regular
    del latents_step_regular, noise_pred_regular
    gc.collect(); torch.cuda.empty_cache()

del latents_step, noise_pred
torch.cuda.empty_cache()
print(f'\n✅ CLIP-MMD DPS Complete! {len(step_vis_data)} steps stored.')

##  Final Results

In [ ]:
# CHANGED: new cell — compute final MMD for both paths
print('Computing final MMD (regular)...')
regular_mmd, regular_eval_photos, _ = evaluate_distribution_mmd(
    latents_regular, architect.vae, architect.image_processor,
    sprinter, clip_model, clip_processor,
    all_clip_embeddings,eval_prompt=sprinter_eval_prompt, n_eval=n_eval, device=device,
)

print('Computing final MMD (DPS)...')
dps_mmd, dps_eval_photos, _ = evaluate_distribution_mmd(
    latents, architect.vae, architect.image_processor,
    sprinter, clip_model, clip_processor,
    all_clip_embeddings, eval_prompt=sprinter_eval_prompt, n_eval=n_eval, device=device,
)

print(f'Regular MMD : {regular_mmd:.6f}')
print(f'DPS MMD     : {dps_mmd:.6f}')
print(f'Delta (↓ better for DPS): {regular_mmd - dps_mmd:.6f}')

In [ ]:
# CHANGED: new cell — log all final results to wandb
with torch.no_grad():
    final_dps_pil     = latent_to_pil(latents,         architect.vae, architect.image_processor)
    final_regular_pil = latent_to_pil(latents_regular, architect.vae, architect.image_processor)

wandb.log({
    'final_dps_mmd':            dps_mmd,
    'final_regular_mmd':        regular_mmd,
    'mmd_delta':                regular_mmd - dps_mmd,
    'mmd_relative_improvement': (regular_mmd - dps_mmd) / (regular_mmd + 1e-8),
    'final_scribble_dps':       wandb.Image(final_dps_pil),
    'final_scribble_regular':   wandb.Image(final_regular_pil),
    'dps_eval_photos':          [wandb.Image(p) for p in dps_eval_photos],
    'regular_eval_photos':      [wandb.Image(p) for p in regular_eval_photos],
})

wandb.summary['final_dps_mmd']     = dps_mmd
wandb.summary['final_regular_mmd'] = regular_mmd
wandb.summary['mmd_delta']         = regular_mmd - dps_mmd

wandb.finish()
print('✅ wandb logging complete.')

In [ ]:
# CHANGED: new cell — side by side photo comparison
plot_row(regular_eval_photos, f'Regular final photos  (MMD={regular_mmd:.4f})')
plot_row(dps_eval_photos,     f'DPS final photos      (MMD={dps_mmd:.4f})')

In [ ]:
# unchanged
steps      = [d['step']          for d in step_gradients]
mmd_vals   = [d['mmd_loss']      for d in step_gradients]
grad_norms = [d['gradient_norm'] for d in step_gradients]
zetas      = [d['zeta_i']        for d in step_gradients]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CLIP-MMD DPS Training Curves', fontsize=14, fontweight='bold')
axes[0].plot(steps, mmd_vals,   color='royalblue'); axes[0].set_title('MMD Loss');      axes[0].set_xlabel('Step'); axes[0].grid(True, alpha=0.3)
axes[1].plot(steps, grad_norms, color='crimson');   axes[1].set_title('Gradient Norm'); axes[1].set_xlabel('Step'); axes[1].grid(True, alpha=0.3)
axes[2].plot(steps, zetas,      color='seagreen');  axes[2].set_title('Zeta (ζ)');      axes[2].set_xlabel('Step'); axes[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()